# Appendix A Computational Lab
## Measure Theory and Stieltjes Integration

This notebook accompanies Appendix A of *Probability Theory with Python and AI*.

The appendix supplies the measure-theoretic foundation behind the probability notation used throughout the book:

$$
\boxed{
\text{probability measure}
\longrightarrow
\text{Lebesgue integral}
\longrightarrow
\text{expectation}.
}
$$

It also explains the distribution-level bridge

$$
\boxed{
F_X
\longrightarrow
P_X(dx)=dF_X(x)
\longrightarrow
E[g(X)]
=
\int g\,dF_X.
}
$$

### Learning goals

By the end of the lab you should be able to:

1. work with measures, null sets, almost-everywhere statements and $\sigma$-finiteness;
2. use continuity from below and continuity from above;
3. distinguish Borel and Lebesgue measure;
4. test measurability using threshold events and Borel inverse images;
5. integrate simple functions;
6. construct the nonnegative Lebesgue integral by supremum over simple minorants;
7. use positive and negative parts for signed integrals;
8. understand $L^1$ and why $\infty-\infty$ is not allowed;
9. apply Monotone Convergence, Fatou and Dominated Convergence;
10. use product measures and Tonelli--Fubini;
11. understand absolute continuity of measures and the Radon--Nikodym derivative;
12. identify expectation as Lebesgue integration with respect to $P$;
13. use pushforward/distribution measures and LOTUS;
14. interpret $\int g\,dF_X$ as a Lebesgue--Stieltjes integral;
15. compute atomic and density cases from the same Stieltjes notation;
16. define the classical Riemann--Stieltjes integral from tagged sums;
17. use the continuous-integrand/increasing-integrator existence theorem;
18. reduce smooth and absolutely continuous integrators to ordinary weighted integrals;
19. compute step-integrator Stieltjes integrals;
20. understand the agreement theorem between Riemann-- and Lebesgue--Stieltjes integrals;
21. explain why classical Riemann--Stieltjes integration is not a complete foundation for expectation;
22. distinguish principal values from expectations;
23. compare Riemann and Lebesgue integration for continuous functions;
24. use Stieltjes integration by parts;
25. derive layer-cake and positive-part formulas;
26. understand why Dirichlet's function motivates Lebesgue integration;
27. audit AI-generated measure-theoretic claims.

> **Foundational hierarchy.** Expectation is defined on the sample space through the Lebesgue integral. Integration against $P_X$ or $F_X$ is then a change-of-variables representation.


## 0. Setup


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def simple_integral(values, masses):
    return sum(v*m for v,m in zip(values,masses))


def dyadic_floor(x, n, cap=None):
    x = np.asarray(x,dtype=float)
    y = np.floor((2**n)*x)/(2**n)
    if cap is not None:
        y = np.minimum(y,cap)
    return y


def rs_sum(f, alpha, partition, tags):
    return sum(
        f(tags[i])
        *
        (
            alpha(partition[i+1])
            -
            alpha(partition[i])
        )
        for i in range(len(tags))
    )


def cauchy_positive_truncation(R):
    return math.log(1+R*R)/(2*math.pi)


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'><b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


## 1. Measures and probability spaces

A measure space is

$$
(\Omega,\mathcal F,\mu),
$$

where $\mathcal F$ is a $\sigma$-algebra and

$$
\mu:\mathcal F\to[0,\infty]
$$

satisfies

$$
\mu(\varnothing)=0
$$

and countable additivity on pairwise disjoint sets.

A probability measure is a measure with

$$
\boxed{
\mu(\Omega)=1.
}
$$


### Null sets, almost everywhere and $\sigma$-finiteness

A measurable set $N$ is null if

$$
\mu(N)=0.
$$

A property holds $\mu$-almost everywhere when it fails only on a null set.

On a probability space, “$P$-almost everywhere” means “almost surely.”

A measure is $\sigma$-finite if

$$
\boxed{
\Omega
=
\bigcup_{n=1}^{\infty}E_n,
\qquad
\mu(E_n)<\infty.
}
$$


### Basic measure properties

For measurable sets,

$$
A\subseteq B
\Longrightarrow
\mu(A)\le\mu(B),
$$

and

$$
\boxed{
\mu\left(
\bigcup_nA_n
\right)
\le
\sum_n\mu(A_n).
}
$$

If $A\subseteq B$ and $\mu(A)<\infty$,

$$
\mu(B\setminus A)
=
\mu(B)-\mu(A).
$$


### Continuity of measures

If

$$
A_n\uparrow A,
$$

then

$$
\boxed{
\mu(A_n)\to\mu(A).
}
$$

If

$$
A_n\downarrow A
$$

and

$$
\mu(A_1)<\infty,
$$

then

$$
\boxed{
\mu(A_n)\to\mu(A).
}
$$

The finiteness condition in continuity from above cannot be dropped.


In [ ]:
cont_n = widgets.IntSlider(value=10,min=1,max=100,description="n")
cont_output = widgets.Output()

def update_continuity(*_):
    with cont_output:
        clear_output(wait=True)
        n = cont_n.value

        # Under standard normal probability, approximate P((-n,n)).
        # erf gives the exact cdf relation.
        p = math.erf(n/math.sqrt(2))

        display(Math(r"P((-n,n))=" + f"{p:.12f}"))
        display(Math(r"\lim_{n\to\infty}P((-n,n))=1"))

cont_n.observe(update_continuity,names="value")
display(widgets.VBox([cont_n,cont_output]))
update_continuity()


## 2. Borel sets and Lebesgue measure

The Borel $\sigma$-algebra

$$
\boxed{
\mathcal B(\mathbb R)
}
$$

is the smallest $\sigma$-algebra containing every open interval.

Lebesgue measure $m$ satisfies

$$
\boxed{
m((a,b])=b-a.
}
$$

On Borel sets this determines the usual length measure.


Every countable subset of $\mathbb R$ has Lebesgue measure zero.

In particular,

$$
\boxed{
m(\mathbb Q\cap[0,1])=0.
}
$$

This fact is essential in the historical Dirichlet-function example.


## 3. Measurable functions

A function

$$
f:\Omega\to\mathbb R
$$

is measurable when

$$
\boxed{
\{f\le x\}\in\mathcal F
}
$$

for every real $x$.


Equivalent tests use

$$
\{f<x\},
\qquad
\{f>x\},
$$

or, most generally,

$$
\boxed{
f^{-1}(B)\in\mathcal F
}
$$

for every Borel set $B$.


If $f,g$ are measurable, then so are

$$
f+g,
\quad
fg,
\quad
|f|,
\quad
\max(f,g),
\quad
\min(f,g).
$$

If $h$ is continuous, then

$$
\boxed{
h\circ f
}
$$

is measurable.


## 4. Simple functions

A nonnegative simple function has a canonical positive-level representation

$$
\boxed{
s
=
\sum_{k=1}^{m}
a_k\mathbf1_{A_k},
\qquad
a_k>0,
}
$$

where the $A_k$ are pairwise disjoint measurable level sets.

The zero level is omitted.


Its integral is

$$
\boxed{
\int s\,d\mu
=
\sum_{k=1}^{m}
a_k\mu(A_k).
}
$$

This is exactly the finite-valued expectation formula when $\mu=P$.


In [ ]:
values = [1,3,5]
masses = [0.2,0.5,0.3]

display(Math(
    r"\int s\,dP="
    + f"{simple_integral(values,masses):.6f}"
))


For nonnegative simple functions, integration is additive, positively homogeneous and monotone.


## 5. Lebesgue integral of a nonnegative function

If

$$
f:\Omega\to[0,\infty]
$$

is measurable, define

$$
\boxed{
\int f\,d\mu
=
\sup
\left\{
\int s\,d\mu:
0\le s\le f,\ 
s\text{ simple}
\right\}.
}
$$

The value may equal $+\infty$.


### Approximation by simple functions

Every nonnegative measurable function admits simple approximations

$$
\boxed{
0\le s_1\le s_2\le\cdots\le f,
\qquad
s_n\uparrow f.
}
$$

This is the technical bridge from finite-valued integrands to the general nonnegative integral.


In [ ]:
approx_n = widgets.IntSlider(value=3,min=1,max=8,description="n")
approx_output = widgets.Output()

def update_simple_approx(*_):
    with approx_output:
        clear_output(wait=True)

        n = approx_n.value
        x = np.linspace(0,2,800)
        f = np.exp(x/2)
        s = dyadic_floor(f,n,cap=n)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(x,f,label="f")
        ax.step(x,s,where="post",label="simple minorant")
        ax.legend()
        ax.set_title("Dyadic simple approximation")
        plt.show()

        display(Markdown(
            f"Maximum grid gap: **{np.max(f-s):.6f}**"
        ))

approx_n.observe(update_simple_approx,names="value")
display(widgets.VBox([approx_n,approx_output]))
update_simple_approx()


## 6. Signed integrals and $L^1$

For a real-valued measurable function,

$$
f^+=\max(f,0),
$$

$$
f^-=\max(-f,0),
$$

and

$$
f=f^+-f^-.
$$

The extended integral is

$$
\boxed{
\int f\,d\mu
=
\int f^+\,d\mu
-
\int f^-\,d\mu
}
$$

whenever at least one of the two nonnegative integrals is finite.

If both are infinite, the integral is undefined.


The function is integrable when

$$
\boxed{
\int|f|\,d\mu<\infty.
}
$$

This is the space

$$
\boxed{
L^1(\mu).
}
$$

For integrable functions, both positive and negative parts have finite integrals.


## 7. Monotone Convergence Theorem

If

$$
0\le f_1\le f_2\le\cdots
$$

and

$$
f_n\uparrow f
$$

pointwise, then

$$
\boxed{
\int f_n\,d\mu
\uparrow
\int f\,d\mu.
}
$$

The limiting integral is allowed to be infinite.


In [ ]:
mct_n = widgets.IntSlider(value=4,min=1,max=12,description="n")
mct_output = widgets.Output()

def update_mct(*_):
    with mct_output:
        clear_output(wait=True)
        n = mct_n.value
        value = 1-math.exp(-n)
        display(Math(r"\int_0^n e^{-x}\,dx=" + f"{value:.10f}"))
        display(Math(r"\longrightarrow1"))

mct_n.observe(update_mct,names="value")
display(widgets.VBox([mct_n,mct_output]))
update_mct()


## 8. Fatou's lemma

For nonnegative measurable $f_n$,

$$
\boxed{
\int
\liminf_{n\to\infty}f_n
\,d\mu
\le
\liminf_{n\to\infty}
\int f_n\,d\mu.
}
$$

Fatou is a one-sided inequality and is often the key ingredient in dominated-convergence arguments.


## 9. Dominated Convergence Theorem

If

$$
f_n\to f
$$

almost everywhere and there exists $g\in L^1(\mu)$ such that

$$
|f_n|\le g
$$

almost everywhere, then

$$
\boxed{
\int f_n\,d\mu
\to
\int f\,d\mu.
}
$$

The dominating function supplies the missing control that pointwise convergence alone does not provide.


### Pointwise convergence is not enough

On $[0,1]$, let

$$
f_n(x)
=
n\mathbf1_{(0,1/n)}(x).
$$

Then

$$
f_n(x)\to0
$$

for every fixed $x>0$, but

$$
\boxed{
\int_0^1f_n(x)\,dx=1
}
$$

for every $n$.


In [ ]:
spike_n = widgets.IntSlider(value=20,min=1,max=100,description="n")
spike_output = widgets.Output()

def update_spike(*_):
    with spike_output:
        clear_output(wait=True)
        n = spike_n.value
        x = np.linspace(0,1,1000)
        f = np.where((x>0)&(x<1/n),n,0)

        fig, ax = plt.subplots(figsize=(8,3.2))
        ax.plot(x,f)
        ax.set_title("Moving spike: pointwise convergence without integral convergence")
        plt.show()

        display(Math(r"\int_0^1f_n\,dx=1"))

spike_n.observe(update_spike,names="value")
display(widgets.VBox([spike_n,spike_output]))
update_spike()


## 10. Product measures

For $\sigma$-finite measure spaces, the product measure

$$
\boxed{
\mu\otimes\nu
}
$$

is characterized on measurable rectangles by

$$
\boxed{
(\mu\otimes\nu)(A\times B)
=
\mu(A)\nu(B).
}
$$

For one-dimensional Lebesgue measure,

$$
m\otimes m
$$

is planar Lebesgue measure.


## 11. Tonelli--Fubini

If

$$
f:S\times T\to[0,\infty]
$$

is product-measurable, Tonelli gives

$$
\boxed{
\int f\,d(\mu\otimes\nu)
=
\int_S\int_Tf(x,y)\,d\nu(y)\,d\mu(x)
=
\int_T\int_Sf(x,y)\,d\mu(x)\,d\nu(y).
}
$$

The common value may be $+\infty$.


For signed $f$, Fubini applies under absolute integrability:

$$
\boxed{
\int|f|\,d(\mu\otimes\nu)<\infty.
}
$$

Then the iterated integrals are finite and the order can be exchanged.


### Triangle example

For

$$
f(x,y)
=
\mathbf1_{\{0<y<x<1\}},
$$

Tonelli gives

$$
\int_0^1\int_0^x1\,dy\,dx
=
\frac12
=
\int_0^1\int_y^11\,dx\,dy.
$$


In [ ]:
# Numerical triangle area.
N = 800
x = np.linspace(0,1,N)
y = np.linspace(0,1,N)
X,Y = np.meshgrid(x,y,indexing="ij")
indicator = (Y < X).astype(float)

area = indicator.mean()
display(Math(r"\text{grid area approximation}\approx" + f"{area:.6f}"))


## 12. Radon--Nikodym theorem

If $\mu,\nu$ are $\sigma$-finite positive measures and

$$
\nu\ll\mu,
$$

then there is a measurable density

$$
\boxed{
\frac{d\nu}{d\mu}
}
$$

such that

$$
\boxed{
\nu(A)
=
\int_A
\frac{d\nu}{d\mu}
\,d\mu.
}
$$

The density is unique $\mu$-almost everywhere.


### Example

On $[0,1]$, define

$$
\nu(A)
=
\int_A2x\,dx.
$$

Then

$$
\boxed{
\frac{d\nu}{dm}(x)=2x
}
$$

almost everywhere.


In [ ]:
grid = np.linspace(0,1,100000)
density = 2*grid

if hasattr(np,"trapezoid"):
    total = np.trapezoid(density,grid)
else:
    total = np.trapz(density,grid)

display(Math(r"\nu([0,1])\approx" + f"{total:.8f}"))


## 13. Expectation as a Lebesgue integral

On a probability space,

$$
\boxed{
E[X]
=
\int_\Omega X\,dP.
}
$$

For a nonnegative random variable, the value may be $+\infty$.

For real $X$, the expectation is defined through $X^+-X^-$ whenever at least one side is finite.

The random variable is integrable precisely when

$$
\boxed{
E|X|<\infty.
}
$$


MCT and DCT immediately become convergence theorems for expectations:

$$
0\le X_n\uparrow X
\Longrightarrow
E[X_n]\uparrow E[X],
$$

and

$$
X_n\to X,\quad |X_n|\le Y,\quad E|Y|<\infty
\Longrightarrow
E[X_n]\to E[X].
$$


## 14. Distribution measure and pushforward

For a random variable

$$
X:(\Omega,\mathcal F,P)\to\mathbb R,
$$

its law is

$$
\boxed{
P_X(B)
=
P(X\in B).
}
$$

This is the pushforward of $P$ by $X$.


For Borel measurable $g$, if $g\ge0$ or $g(X)$ is integrable,

$$
\boxed{
E[g(X)]
=
\int_{\mathbb R}
g(x)\,P_X(dx).
}
$$

This is the rigorous law-integral form of LOTUS.


In [ ]:
# Bernoulli pushforward example.
p = 0.3
g0 = 0**2 + 1
g1 = 1**2 + 1
law_integral = (1-p)*g0 + p*g1

display(Math(r"\int g\,dP_X=" + f"{law_integral:.6f}"))
display(Math(r"E[X^2+1]=1+p=" + f"{1+p:.6f}"))


## 15. Lebesgue--Stieltjes integration

For a random variable with cdf $F_X$, the associated Lebesgue--Stieltjes measure is exactly $P_X$.

Informally,

$$
\boxed{
P_X(dx)=dF_X(x).
}
$$

This is measure notation, not an ordinary differential.


Define

$$
\boxed{
\int_{\mathbb R}
g(x)\,dF_X(x)
:=
\int_{\mathbb R}
g(x)\,P_X(dx).
}
$$

Therefore

$$
\boxed{
E[g(X)]
=
\int_{\mathbb R}g(x)\,dF_X(x).
}
$$


### Discrete case

If

$$
P(X=x_k)=p_k,
$$

then

$$
\boxed{
\int g\,dF_X
=
\sum_kg(x_k)p_k.
}
$$

Thus the Stieltjes notation contains the ordinary probability-weighted sum.


### Density case

If

$$
P_X(dx)=f_X(x)\,dx,
$$

then

$$
\boxed{
\int g\,dF_X
=
\int g(x)f_X(x)\,dx.
}
$$

The discrete sum and density integral are computational forms of one law integral.


### Mixed distribution

For example,

$$
P_X
=
\frac13\delta_0
+
\frac23\,U(0,1),
$$

so

$$
\boxed{
\int g\,dF_X
=
\frac13g(0)
+
\frac23
\int_0^1g(x)\,dx.
}
$$

No separate concept of expectation is needed.


## 16. Classical Riemann--Stieltjes integral

Let

$$
a=x_0<\cdots<x_n=b
$$

be a partition with tags

$$
\xi_i\in[x_{i-1},x_i].
$$

The tagged Riemann--Stieltjes sum is

$$
\boxed{
S(f,\alpha;P,\xi)
=
\sum_i
f(\xi_i)
[
\alpha(x_i)-\alpha(x_{i-1})
].
}
$$


If all sufficiently fine tagged sums converge to a common value independent of tags and partitions, that value is

$$
\boxed{
\int_a^bf\,d\alpha.
}
$$


### Existence theorem

If $f$ is continuous on $[a,b]$ and $\alpha$ is increasing, then

$$
\boxed{
\int_a^bf\,d\alpha
}
$$

exists in the Riemann--Stieltjes sense.

The quantitative control is supplied by the modulus of continuity of $f$.


In [ ]:
rs_n = widgets.IntSlider(value=10,min=2,max=100,description="subintervals")
rs_output = widgets.Output()

def update_rs(*_):
    with rs_output:
        clear_output(wait=True)
        n = rs_n.value
        partition = np.linspace(0,1,n+1)
        tags = (partition[:-1]+partition[1:])/2

        value = rs_sum(
            lambda x:x,
            lambda x:x*x,
            partition,
            tags,
        )

        display(Math(r"\sum \xi_i\Delta(x^2)\approx" + f"{value:.10f}"))
        display(Math(r"\int_0^1x\,d(x^2)=\frac23"))

rs_n.observe(update_rs,names="value")
display(widgets.VBox([rs_n,rs_output]))
update_rs()


## 17. Smooth and absolutely continuous integrators

If $\alpha\in C^1([a,b])$ and $f$ is continuous,

$$
\boxed{
\int_a^b
f(x)\,d\alpha(x)
=
\int_a^b
f(x)\alpha'(x)\,dx.
}
$$


More generally, if $\alpha$ is absolutely continuous, then $\alpha'$ exists almost everywhere,

$$
\alpha'\in L^1,
$$

and

$$
\boxed{
\alpha(x)
=
\alpha(a)
+
\int_a^x\alpha'(u)\,du.
}
$$

For continuous $f$,

$$
\boxed{
\int_a^bf\,d\alpha
=
\int_a^bf(x)\alpha'(x)\,dx.
}
$$


Absolute continuity does **not** imply that the derivative exists everywhere or is continuous.


## 18. Step integrators and atoms

If an increasing integrator has jumps of sizes $w_j$ at points $c_j$, then for continuous $f$,

$$
\boxed{
\int f\,d\alpha
=
\sum_jw_jf(c_j)
}
$$

over an interval containing the jumps.

This is exactly the algebra of a discrete probability law.


In [ ]:
points = np.array([0,1,2],dtype=float)
weights = np.array([0.25,0.5,0.25])
value = np.sum(weights*points**2)

display(Math(r"\int x^2\,d\alpha=" + f"{value:.6f}"))


## 19. Agreement of the two Stieltjes integrals

Let $f$ be continuous on $[a,b]$ and let $\alpha$ be increasing and right-continuous.

Associate the measure

$$
\mu_\alpha((u,v])
=
\alpha(v)-\alpha(u).
$$

Then

$$
\boxed{
\int_a^bf(x)\,d\alpha(x)
=
\int_{(a,b]}
f(x)\,\mu_\alpha(dx).
}
$$

Thus classical tagged sums and the measure-theoretic Stieltjes integral agree under the stated hypotheses.


### Endpoint convention

The classical increment

$$
\alpha(x_i)-\alpha(x_{i-1})
$$

naturally corresponds to the measure of

$$
(x_{i-1},x_i].
$$

This matters when the integrator has an atom exactly at the left endpoint.


### Expectation from a cdf

For continuous $g$ on a finite interval,

$$
\int_a^bg(x)\,dF_X(x)
=
\int_{(a,b]}g(x)\,P_X(dx).
$$

If $g(X)$ is integrable, the improper Lebesgue--Stieltjes interpretation gives

$$
\boxed{
E[g(X)]
=
\int_{-\infty}^{\infty}
g(x)\,dF_X(x).
}
$$


## 20. Why Riemann--Stieltjes is not a complete foundation for expectation

The fixed-integrator identity

$$
\int
(cf+dh)\,d\alpha
=
c\int f\,d\alpha
+
d\int h\,d\alpha
$$

is true whenever the relevant classical Stieltjes integrals exist.

But it does not by itself prove

$$
E[cX+dY]
=
cE[X]+dE[Y].
$$

The distribution-based representations involve three different integrators:

$$
F_X,
\qquad
F_Y,
\qquad
F_{cX+dY}.
$$

The law of the sum depends on the joint law, not merely on the marginals.


### Discontinuous payoff at an atom

Let $X=0$ surely and

$$
g=\mathbf1_{\{0\}}.
$$

Then

$$
E[g(X)]=1.
$$

The Lebesgue--Stieltjes integral against $\delta_0$ equals one.

But the classical Riemann--Stieltjes tagged sums can be forced to take different values by choosing the tag at or away from the common discontinuity.

So the classical integral need not exist even though the probabilistic expectation is completely well defined.


### A principal value is not an expectation

For standard Cauchy $X$,

$$
\int_{-R}^{R}x\,dF_X(x)=0
$$

for every $R$ by symmetry.

But

$$
E[X^+]=\infty,
\qquad
E[X^-]=\infty.
$$

Hence

$$
\boxed{
E[X]\text{ does not exist}.
}
$$

The symmetric value zero is only a Cauchy principal value.


In [ ]:
pv_R = widgets.FloatSlider(value=10,min=1,max=100,step=1,description="R")
pv_output = widgets.Output()

def update_cauchy_parts(*_):
    with pv_output:
        clear_output(wait=True)
        R = pv_R.value
        positive = cauchy_positive_truncation(R)
        display(Math(r"\int_0^R\frac{x}{\pi(1+x^2)}\,dx=" + f"{positive:.6f}"))
        display(Math(r"\longrightarrow\infty\quad(R\to\infty)"))

pv_R.observe(update_cauchy_parts,names="value")
display(widgets.VBox([pv_R,pv_output]))
update_cauchy_parts()


The logical hierarchy is therefore

$$
\boxed{
E[Z]
=
\int_\Omega Z\,dP
}
$$

first, then

$$
\boxed{
E[g(X)]
=
\int g\,dP_X
=
\int g\,dF_X.
}
$$

Classical Riemann--Stieltjes sums provide an additional computational representation only when their extra regularity hypotheses are satisfied.


## 21. Riemann and Lebesgue integration

If $f$ is continuous on $[a,b]$, its Riemann and Lebesgue integrals agree:

$$
\boxed{
\int_a^bf(x)\,dx\big|_{\mathrm{Riemann}}
=
\int_{[a,b]}f(x)\,dx\big|_{\mathrm{Lebesgue}}.
}
$$


## 22. Stieltjes integration by parts

Assume $f,\alpha$ are continuous and the two Stieltjes integrals exist.

Then

$$
\boxed{
\int_a^bf\,d\alpha
+
\int_a^b\alpha\,df
=
f(b)\alpha(b)-f(a)\alpha(a).
}
$$

This identity transfers variation between the integrand and integrator.


## 23. Layer cake and positive parts

If $X\ge0$,

$$
\boxed{
E[X]
=
\int_0^\infty
P(X>t)\,dt,
}
$$

with both sides possibly infinite.


For $d\ge0$,

$$
\boxed{
E[(X-d)^+]
=
\int_d^\infty
P(X>x)\,dx.
}
$$

For $u>0$,

$$
\boxed{
E[X\wedge u]
=
\int_0^u
P(X>t)\,dt.
}
$$


In [ ]:
# Exponential layer-cake check.
beta = 0.5
grid = np.linspace(0,30,100000)
surv = np.exp(-beta*grid)

if hasattr(np,"trapezoid"):
    approx = np.trapezoid(surv,grid)
else:
    approx = np.trapz(surv,grid)

display(Math(r"\int_0^\infty e^{-0.5t}\,dt=2"))
display(Math(r"\text{numerical truncation}\approx" + f"{approx:.8f}"))


## 24. Historical problem: Dirichlet's function

On $[0,1]$, define

$$
f(x)
=
\mathbf1_{\mathbb Q}(x).
$$

Every interval contains both rational and irrational points.

Therefore every lower Riemann sum is zero and every upper Riemann sum is one.

So $f$ is not Riemann integrable.


However,

$$
m(\mathbb Q\cap[0,1])=0,
$$

so the Lebesgue integral is

$$
\boxed{
\int_0^1
\mathbf1_{\mathbb Q}(x)\,dx
=
0.
}
$$

This is a compact illustration of why the Lebesgue theory is more flexible than Riemann integration.


## 25. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Measure","measure"),
        ("MCT","mct"),
        ("DCT","dct"),
        ("LOTUS","lotus"),
        ("Stieltjes","stieltjes"),
        ("Atom","atom"),
        ("Layer cake","layer"),
        ("Principal value","pv"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}

def make_exercise(_=None):
    kind = exercise_kind.value
    if kind == "random":
        kind = exercise_rng.choice(["measure","mct","dct","lotus","stieltjes","atom","layer","pv"])

    if kind == "measure":
        target = "yes"
        prompt = "Is every probability measure sigma-finite? yes/no"
        hint = "A probability measure is finite."
        solution = r"\text{Yes.}"

    elif kind == "mct":
        target = "yes"
        prompt = "May the MCT limit integral be +infinity? yes/no"
        hint = "The theorem is stated in the extended nonnegative sense."
        solution = r"\text{Yes.}"

    elif kind == "dct":
        target = "no"
        prompt = "Does pointwise convergence alone imply convergence of integrals? yes/no"
        hint = "Recall the moving-spike example."
        solution = r"\text{No.}"

    elif kind == "lotus":
        target = "yes"
        prompt = "For Borel g with g>=0, is E[g(X)]=integral g dP_X? yes/no"
        hint = "This is the pushforward change-of-variables theorem."
        solution = r"\text{Yes.}"

    elif kind == "stieltjes":
        target = str(2/3)
        prompt = "Compute integral_0^1 x d(x^2) as a decimal."
        hint = "Use d(x^2)=2x dx."
        solution = r"2/3."

    elif kind == "atom":
        target = "1.5"
        prompt = "An integrator has masses 1/4,1/2,1/4 at 0,1,2. Compute integral x^2 d alpha."
        hint = "Use the weighted discrete sum."
        solution = r"3/2."

    elif kind == "layer":
        target = "2"
        prompt = "If X~Exp(rate 0.5), compute integral_0^infinity P(X>t) dt."
        hint = "It equals E[X]."
        solution = r"2."

    else:
        target = "no"
        prompt = "Is a symmetric Cauchy principal value automatically an expectation? yes/no"
        hint = "Positive and negative parts may both be infinite."
        solution = r"\text{No.}"

    state.clear()
    state.update(target=target,hint=hint,solution=solution)
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n"+prompt))
    with feedback_output:
        clear_output(wait=True)

def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** "+state["hint"]))

def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))

def check(_):
    with feedback_output:
        clear_output(wait=True)
        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")
        correct = guess == target
        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 1e-4
            except Exception:
                pass
        display(Markdown("**Correct.**" if correct else "**Not yet. Check the exact theorem hypotheses.**"))

new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))
make_exercise()


## 26. AI Audit: measure and Stieltjes claims

Audit the following statements.

1. “If $f_n\to f$ pointwise, then $\int f_n\,d\mu\to\int f\,d\mu$.”
2. “Because $F_X$ is a function, $dF_X(x)$ is simply $F_X'(x)\,dx$.”
3. “A symmetric principal value of $\int x\,dF_X(x)$ is always the expectation.”
4. “For $\sigma$-finite measures, product measure is determined on rectangles, so Tonelli may be applied to every nonnegative product-measurable function.”
5. “If $\alpha$ is absolutely continuous, then its derivative exists everywhere and is continuous.”

The repairs are:

- pointwise convergence alone is insufficient; MCT, DCT or another valid theorem is needed;
- $dF_X$ denotes the distribution measure and may contain atoms or singular parts;
- expectation forbids $\infty-\infty$ cancellation;
- product-measure construction and Tonelli's subsequent integration theorem are conceptually distinct steps;
- absolute continuity gives an a.e. derivative in $L^1$, not an everywhere continuous derivative.


### Suggested AI-guided activities

- Compare $\sum_xg(x)P(X=x)$, $\int g(x)f_X(x)\,dx$ and $\int g(x)\,dF_X(x)$ and construct a mixed distribution.
- Ask for a proposed proof of DCT and check whether integrability of the limit is established.
- Compare three Stieltjes integrators: $\alpha(x)=x$, a smooth increasing integrator and a step integrator.
- Ask for a counterexample to “pointwise convergence implies convergence of integrals.”


## 27. Self-check quiz


In [ ]:
quiz_data = [
    ("1. Every probability measure is finite:", ["Choose...","true","false"], "true", r"P(\Omega)=1."),
    ("2. Countable subsets of R have Lebesgue measure zero:", ["Choose...","true","false"], "true", r"m(A)=0\text{ for countable }A."),
    ("3. Pointwise convergence alone implies convergence of integrals:", ["Choose...","true","false"], "false", r"\text{Additional hypotheses are required.}"),
    ("4. MCT allows an infinite limiting integral:", ["Choose...","true","false"], "true", r"\int f_n\uparrow\int f\text{ in }[0,\infty]."),
    ("5. Tonelli is for nonnegative measurable functions:", ["Choose...","true","false"], "true", r"\text{The common value may be }+\infty."),
    ("6. Fubini for signed functions uses absolute integrability:", ["Choose...","true","false"], "true", r"\int|f|<\infty."),
    ("7. dF_X is always F_X' dx:", ["Choose...","true","false"], "false", r"dF_X=P_X\text{ as a measure.}"),
    ("8. A continuous integrand and increasing integrator give an RS integral:", ["Choose...","true","false"], "true", r"\int f\,d\alpha\text{ exists.}"),
    ("9. Absolutely continuous alpha must have continuous derivative:", ["Choose...","true","false"], "false", r"\alpha'\text{ exists a.e. and is }L^1."),
    ("10. A Cauchy principal value can exist when E[X] does not:", ["Choose...","true","false"], "true", r"\text{Standard Cauchy is the example.}"),
]

quiz_widgets = []
rows = []
for prompt,options,_,_ in quiz_data:
    d = widgets.Dropdown(options=options,value="Choose...",layout=widgets.Layout(width="480px"))
    quiz_widgets.append(d)
    rows.append(widgets.HBox([widgets.HTML(f"<div style='width:720px'>{prompt}</div>"),d]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()

def grade(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(w.value == correct for w,(_,_,correct,_) in zip(quiz_widgets,quiz_data))
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i,(w,(_,_,correct,explanation)) in enumerate(zip(quiz_widgets,quiz_data),1):
            mark = "✓" if w.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** correct answer = `{correct}`"))
            display(Math(explanation))

grade_button.on_click(grade)
display(widgets.VBox(rows+[grade_button,quiz_output]))


## 28. Automatic mathematical verification


In [ ]:
# Simple integral.
assert abs(simple_integral([1,3,5],[0.2,0.5,0.3])-3.2) < 1e-12

# MCT example.
assert abs((1-math.exp(-20))-1) < 1e-8

# Bernoulli pushforward.
p = 0.3
assert abs((1-p)*1+p*2-(1+p)) < 1e-12

# Riemann-Stieltjes smooth example.
partition = np.linspace(0,1,10001)
tags = (partition[:-1]+partition[1:])/2
value = rs_sum(lambda x:x,lambda x:x*x,partition,tags)
assert abs(value-2/3) < 1e-7

# Atomic Stieltjes example.
points = np.array([0,1,2],dtype=float)
weights = np.array([0.25,0.5,0.25])
assert abs(np.sum(weights*points**2)-1.5) < 1e-12

# Layer cake exponential.
assert abs(1/0.5-2) < 1e-12

# Cauchy positive part grows with truncation.
assert cauchy_positive_truncation(100) > cauchy_positive_truncation(10)

show_result(
    "All Appendix A automatic checks passed",
    r"\int f\,d\mu=\sup_{0\le s\le f,\ s\text{ simple}}\int s\,d\mu",
    r"E[g(X)]=\int g\,dP_X=\int g\,dF_X",
    r"\int_a^bf\,d\alpha=\int_{(a,b]}f\,d\mu_\alpha",
    r"E[X]=\int_0^\infty P(X>t)\,dt\quad(X\ge0)",
)


## 29. Appendix map

| Concept | Computational representation |
|---|---|
| measure | finite examples and continuity of probabilities |
| null sets | countable-set principle |
| measurable functions | threshold/Borel tests |
| simple functions | weighted indicator sums |
| nonnegative integral | simple-minorant approximation |
| signed integral | positive/negative parts |
| MCT | growing exponential interval |
| Fatou | one-sided limit principle |
| DCT | moving-spike warning |
| product measure | rectangle area |
| Tonelli--Fubini | triangle integral |
| Radon--Nikodym | density $2x$ |
| expectation | integral with respect to $P$ |
| pushforward | Bernoulli law example |
| Lebesgue--Stieltjes | unified discrete/density/mixed notation |
| Riemann--Stieltjes | tagged-sum convergence |
| smooth integrator | weighted ordinary integral |
| step integrator | atomic weighted sum |
| agreement theorem | common Stieltjes value |
| limitations | atom discontinuity and Cauchy principal value |
| Riemann versus Lebesgue | agreement for continuous functions |
| integration by parts | Stieltjes identity |
| layer cake | survival integration |
| historical problem | Dirichlet function |
| AI Audit | theorem-hypothesis checking |

The central message is:

$$
\boxed{
\text{Lebesgue integration is the foundation;}
}
$$

$$
\boxed{
\text{Stieltjes notation is the unified law-level computational language.}
}
$$
